In [1]:
import cv2
import numpy as np
from util import get_parking_spots_bboxes, empty_or_not

In [2]:
def calc_diff(im1, im2):
    """Calculate the mean pixel difference between two images."""
    return np.abs(np.mean(im1) - np.mean(im2))

In [3]:
mask_path = './mask_1920_1080.png'
video_path = './data/parking_1920_1080.mp4'

In [4]:
mask = cv2.imread(mask_path, 0)
if mask is None:
    raise FileNotFoundError(f"Mask file not found: {mask_path}")

# Open video
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise FileNotFoundError(f"Video file not found or can't be opened: {video_path}")

In [5]:
connected_components = cv2.connectedComponentsWithStats(mask, 4, cv2.CV_32S)
spots = get_parking_spots_bboxes(connected_components)

In [6]:
spots_status = [None] * len(spots)  
diffs = [None] * len(spots)       
previous_frame = None             
frame_nmr = 0
step = 30    

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        break  

    if frame_nmr % step == 0 and previous_frame is not None:
        for spot_indx, spot in enumerate(spots):
            x1, y1, w, h = spot
            spot_crop = frame[y1:y1 + h, x1:x1 + w, :]
            prev_crop = previous_frame[y1:y1 + h, x1:x1 + w, :]
            diffs[spot_indx] = calc_diff(spot_crop, prev_crop)

        print([diffs[j] for j in np.argsort(diffs)][::-1])

    if frame_nmr % step == 0:
        if previous_frame is None:
            arr_ = range(len(spots))
        else:
            arr_ = [j for j in np.argsort(diffs) if diffs[j] / np.max(diffs) > 0.4]

        for spot_indx in arr_:
            x1, y1, w, h = spots[spot_indx]
            spot_crop = frame[y1:y1 + h, x1:x1 + w, :]
            spots_status[spot_indx] = empty_or_not(spot_crop)

        previous_frame = frame.copy()

    for spot_indx, (x1, y1, w, h) in enumerate(spots):
        color = (0, 255, 0) if spots_status[spot_indx] else (0, 0, 255)
        cv2.rectangle(frame, (x1, y1), (x1 + w, y1 + h), color, 2)

    cv2.rectangle(frame, (80, 20), (550, 80), (0, 0, 0), -1)
    cv2.putText(frame, f'Available spots: {sum(spots_status)} / {len(spots_status)}',
                (100, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    cv2.namedWindow('frame', cv2.WINDOW_NORMAL)
    cv2.imshow('frame', frame)
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

    frame_nmr += 1

cap.release()
cv2.destroyAllWindows()

[7.591036414565821, 7.32528180354268, 7.1615858737298055, 7.014159586873248, 6.439095928226351, 6.401851851851859, 6.110378634212296, 6.088122605363992, 6.0571895424836555, 5.906423331544119, 5.861111111111114, 5.743794769282033, 5.742295518907213, 5.624070317782298, 5.597826086956523, 5.5755772005772, 5.574183006535947, 5.572663139329805, 5.556302521008391, 5.540896218557393, 5.507366035292208, 5.419623521572547, 5.369315342328846, 5.301759834368532, 5.296618357487915, 5.263235294117649, 5.246798029556643, 5.236392914653791, 5.215059137098109, 5.198951994590942, 5.186162870945481, 5.165307971014485, 5.131535947712422, 5.116801893171058, 5.0926015085861, 5.053117380703583, 5.027260179434094, 5.019162640901769, 4.948275862068954, 4.9243156199678, 4.918242296918777, 4.901328273244786, 4.896881287726359, 4.754267310789061, 4.750700280112056, 4.7482492997198875, 4.736815415821496, 4.654303599374032, 4.6359477124183, 4.6250862663906105, 4.598200899550221, 4.5933123249299825, 4.5865905431122